<a href="https://colab.research.google.com/github/rg-smith/remote-sensing-hydro-2026/blob/main/lectures/lecture5-precipitation-data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 5: working with precipitation data

In [ ]:
!pip install geemap

In [ ]:
import ee
import folium
import numpy as np
import branca.colormap as cm
import pandas as pd
import zipfile
import os
from tqdm import tqdm
import requests
import geemap

In [ ]:
#if not ee.data._credentials:
ee.Authenticate()
ee.Initialize(project='replace with your code')

## Define custom functions

In [ ]:
# functions needed for this lab (and some other useful ones that you can use if you're interested)

# to convert a google earth engine image to a python array
def to_array(img,aoi):
  band_arrs = img.sampleRectangle(region=aoi,properties=['scale=1000'],defaultValue=-999)

  band_names=img.bandNames().getInfo()

  for kk in range(len(band_names)):
    if kk==0:
      dat1=np.array(band_arrs.get(band_names[kk]).getInfo())
      dat_full=np.zeros((dat1.shape[0],dat1.shape[1],len(band_names)))
      dat_full[:,:,kk]=dat1
    else:
      dat=np.array(band_arrs.get(band_names[kk]).getInfo())
      dat_full[:,:,kk]=dat
  return(dat_full)

# to calculate an index
def getIndex(image,b1,b2):
  return image.normalizedDifference([b1, b2])

# to calculate a ratio
def getRatio(image1,image2):
  ratio=image1.divide(image2)
  return ratio

# to create a color map from a specific image
def getVisparams(image,aoi,min='',max=''):
  range = image.reduceRegion(ee.Reducer.percentile([1, 99]),aoi,300)
  vals = range.getInfo()
  if min=='':
    min=list(vals.items())[0][1]
  if max=='':
    max=list(vals.items())[1][1]
  visParams = {'min': min, 'max': max, "palette": ["red", "orange", "yellow", "cyan", "blue"]}
  return(visParams)

# to get the link to download an earth engine image
def getLink(image,aoi):
  link = image.getDownloadURL({
    'scale': 1000,
    'crs': 'EPSG:4326',
    'fileFormat': 'GeoTIFF',
    'region': aoi})
  print(link)

# create an earth engine geometry polygon
def addGeometry(min_lon,max_lon,min_lat,max_lat):

  geom = ee.Geometry.Polygon(
      [[[min_lon, max_lat],
        [min_lon, min_lat],
        [max_lon, min_lat],
        [max_lon, max_lat]]])
  return(geom)

def get_center_from_geometry(geom_obj):
  centroid = geom_obj.centroid()
  coords = centroid.getInfo()['coordinates']
  return [coords[1], coords[0]] # Return as [latitude, longitude]

# to export an image to google drive
def export_to_drive(raster,filename,foldername,geometry):
  # Export the image, specifying scale and region.
  task = ee.batch.Export.image.toDrive(**{
      'image': raster,
      'description': filename,
      'folder': foldername,
      'fileNamePrefix': filename,
      'scale': 1000,
      'region': geometry,
      'fileFormat': 'GeoTIFF',
      'formatOptions': {
        'cloudOptimized': 'true'
      },
  })
  task.start()

def get_imgcollection(date1,date2,geometry,collection_name,band_name,function='mean'):
  collection = ee.ImageCollection(collection_name)
  if function=='mean':
      img = collection.filterDate(date1,date2).select(band_name).mean().clip(geometry)
  if function=='sum':
      img = collection.filterDate(date1,date2).select(band_name).sum().clip(geometry)
  return(img)

def get_img(geometry,collection_name,band_name):
  img = ee.Image(collection_name).select(band_name).clip(geometry)
  return(img)

# to get the link to download an earth engine image
def getLink(image,fname,aoi,scale=1000):
  link = image.getDownloadURL({
    'scale': scale,
    'crs': 'EPSG:4326',
    'fileFormat': 'GeoTIFF',
    'region': aoi,
    'name': fname})
  # print(link)
  return(link)

def download_img(img,geom,fname,scale=1000):
    linkname = getLink(img,fname,geom,scale=scale)
    response = requests.get(linkname, stream=True)
    zipped = fname+'.zip'
    with open(zipped, "wb") as handle:
        for data in tqdm(response.iter_content()):
            handle.write(data)

    with zipfile.ZipFile(zipped, 'r') as zip_ref:
        zip_ref.extractall('')
    os.remove(zipped)

def create_reduce_region_function(geometry,
                                  reducer=ee.Reducer.mean(),
                                  scale=1000,
                                  crs='EPSG:4326',
                                  bestEffort=True,
                                  maxPixels=1e13,
                                  tileScale=4):
  """Creates a region reduction function.

  Creates a region reduction function intended to be used as the input function
  to ee.ImageCollection.map() for reducing pixels intersecting a provided region
  to a statistic for each image in a collection. See ee.Image.reduceRegion()
  documentation for more details.

  Args:
    geometry:
      An ee.Geometry that defines the region over which to reduce data.
    reducer:
      Optional; An ee.Reducer that defines the reduction method.
    scale:
      Optional; A number that defines the nominal scale in meters of the
      projection to work in.
    crs:
      Optional; An ee.Projection or EPSG string ('EPSG:5070') that defines
      the projection to work in.
    bestEffort:
      Optional; A Boolean indicator for whether to use a larger scale if the
      geometry contains too many pixels at the given scale for the operation
      to succeed.
    maxPixels:
      Optional; A number specifying the maximum number of pixels to reduce.
    tileScale:
      Optional; A number representing the scaling factor used to reduce
      aggregation tile size; using a larger tileScale (e.g. 2 or 4) may enable
      computations that run out of memory with the default.

  Returns:
    A function that accepts an ee.Image and reduces it by region, according to
    the provided arguments.
  """

  def reduce_region_function(img):
    """Applies the ee.Image.reduceRegion() method.

    Args:
      img:
        An ee.Image to reduce to a statistic by region.

    Returns:
      An ee.Feature that contains properties representing the image region
      reduction results per band and the image timestamp formatted as
      milliseconds from Unix epoch (included to enable time series plotting).
    """

    stat = img.reduceRegion(
        reducer=reducer,
        geometry=geometry,
        scale=scale,
        crs=crs,
        bestEffort=bestEffort,
        maxPixels=maxPixels,
        tileScale=tileScale)

    return ee.Feature(geometry, stat).set({'millis': img.date().millis()})
  return reduce_region_function

# Define a function to transfer feature properties to a dictionary.
def fc_to_dict(fc):
  prop_names = fc.first().propertyNames()
  prop_lists = fc.reduceColumns(
      reducer=ee.Reducer.toList().repeat(prop_names.size()),
      selectors=prop_names).get('list')

  return ee.Dictionary.fromLists(prop_names, prop_lists)

def gee_zonal_mean_img_coll(imageCollection,geometry,scale=1000):
    reduce_iC = create_reduce_region_function(geometry = geometry, scale=scale)
    stat_fc = ee.FeatureCollection(imageCollection.map(reduce_iC)).filter(ee.Filter.notNull(imageCollection.first().bandNames()))
    fc_dict = fc_to_dict(stat_fc).getInfo()

    df = pd.DataFrame(fc_dict)
    df['date'] = pd.to_datetime(df['millis'],unit='ms')
    return(df)

def gee_zonal_mean(date1,date2,geometry,collection_name,band_name,scale=1000):
     imcol = ee.ImageCollection(collection_name).select(band_name).filterDate(date1,date2)
     df = gee_zonal_mean_img_coll(imcol,geometry,scale=scale)
     return(df)

## Load and map precipitation data

In [ ]:
# create a bounding box that defines the study area
geom = addGeometry(-102, -94.6,37,40) # min long, max long, min lat, max lat (kansas)
center = get_center_from_geometry(geom)

# define dates of interest (inclusive).
start = '2019-10-01'
end = '2020-10-01' #can go up to april 2021

# now get gpm precipitation over the same region for a specified time period
#  use get_imgcollection to create variable gpm_img (mean precip)
gpm_img = get_imgcollection(start,end,geom,'NASA/GPM_L3/IMERG_MONTHLY_V07','precipitation','mean').multiply(24*365/12)
gpm_vis_params = getVisparams(gpm_img,geom,min=0,max=100)
gpm_layer_name = 'GPM Precipitation'

# now get prism precipitation over the same time period/region
prism_img = get_imgcollection(start,end,geom,'OREGONSTATE/PRISM/AN81m','ppt','mean')
prism_vis_params = getVisparams(prism_img,geom,min=0,max=100)
prism_layer_name = 'PRISM Precipitation'

In [ ]:
Map = geemap.Map(center=center, zoom=6)
Map.add_basemap("HYBRID")

Map.addLayer(gpm_img,gpm_vis_params,gpm_layer_name)
Map.add_colorbar(gpm_vis_params,label=gpm_layer_name,layer_name=gpm_layer_name)

Map.addLayer(prism_img,prism_vis_params,prism_layer_name)
Map.add_colorbar(prism_vis_params,label=prism_layer_name)

Map.addLayer(geom, {},"Study area")

Map.add_inspector()

Map #visualize the map

## Point-scale analysis
Now, we will compare precipitation products at the Denver International Airport

In [ ]:
# coordinates to DIA
lat = 39.86
lon = -104.68

In [ ]:
import pandas as pd

fc = ee.Geometry.Point([lon,lat])
# use gee_zonal_mean to get precip data from GPM from 2023-01-01 to 2024-12-31
gpm_precip = gee_zonal_mean('2023-01-01','2024-12-31',fc,'NASA/GPM_L3/IMERG_MONTHLY_V07','precipitation')
gpm_weight = gee_zonal_mean('2023-01-01','2024-12-31',fc,'NASA/GPM_L3/IMERG_MONTHLY_V07','gaugeRelativeWeighting')

print(gpm_weight)


In [ ]:
# repeat exercise for prism, chirps
prism_precip = gee_zonal_mean('2023-01-01','2024-12-31',fc,'OREGONSTATE/PRISM/ANm','ppt')
chirps_precip = gee_zonal_mean('2023-01-01','2024-12-31',fc,'UCSB-CHG/CHIRPS/DAILY','precipitation')
chirps_precip_monthly = chirps_precip.groupby(pd.Grouper(key='date', freq='M')).sum()

# convert chirps datetime to first day of month
chirps_precip_monthly['date-mod'] = chirps_precip_monthly.index.strftime('%Y-%m-01')

print(chirps_precip_monthly)

We'll install meteostat, a python package that downloads precipitation data

In [ ]:
# plot data together
import matplotlib.pyplot as plt

plt.figure(figsize=(12,8));
plt.subplot(2,1,1)
plt.plot(gpm_precip['date'],gpm_precip['precipitation']*24*365/12)
plt.plot(prism_precip['date'],prism_precip['ppt'])
plt.plot(pd.to_datetime(chirps_precip_monthly['date-mod']),chirps_precip_monthly['precipitation'])
plt.xlabel('Date')
plt.legend(['GPM','PRISM','CHIRPS'])
plt.ylabel('Precipitation (mm)')
plt.title('Precipitation Comparison')

plt.subplot(2,1,2)
plt.plot(gpm_weight['date'],gpm_weight['gaugeRelativeWeighting'])
plt.xlabel('Date')
plt.ylabel('Gauge Relative Weighting')